# EX - Регрессия Auto MPG

Цель работы — построить регрессионную модель, прогнозирующую расход топлива автомобилей (mpg) на основе их технических характеристик. Используется классический набор данных [Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg) из репозитория UCI Machine Learning, содержащий 398 записей автомобилей 1970–1982 гг.

В ходе работы мы пройдём полный цикл: загрузка и очистка данных, исследовательский анализ с визуализациями, выбор признаков, обучение двух моделей (LinearRegression и Ridge) и их сравнение по метрикам RMSE и R².


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from math import sqrt


## 1. Загрузка данных

Файл `auto-mpg.data` не имеет заголовков и разделён пробелами/табуляцией, поэтому используем `sep=r'\s+'` и задаём имена столбцов вручную. Параметр `na_values='?'` позволяет pandas сразу распознать пропуски в столбце `horsepower` (6 записей содержат `?` вместо числа).

Конструкция `if 'google.colab' in sys.modules` обеспечивает совместимость: в Google Colab данные загружаются с Drive, локально — из текущей директории.


In [ ]:
column_names = ['mpg', 'cylinders', 'displacement', 'horsepower',
                'weight', 'acceleration', 'model_year', 'origin', 'car_name']

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    path = '/content/drive/MyDrive/google_colab/auto-mpg.data'
else:
    path = 'auto-mpg.data'

raw_data = pd.read_csv(path, sep=r'\s+', names=column_names, na_values='?')
raw_data.head()


Набор данных содержит 398 записей автомобилей (1970–1982 гг.) с 9 столбцами:

| Признак | Описание |
|---|---|
| **mpg** | Расход топлива (мили на галлон) — **целевая переменная** |
| **cylinders** | Количество цилиндров (4, 6, 8) |
| **displacement** | Рабочий объём двигателя (куб. дюймы) |
| **horsepower** | Мощность двигателя (л.с.); содержит 6 пропусков (`?`) |
| **weight** | Масса автомобиля (фунты) |
| **acceleration** | Время разгона 0–60 миль/ч (секунды) |
| **model_year** | Год выпуска (70–82) |
| **origin** | Происхождение: 1 = США, 2 = Европа, 3 = Азия |
| **car_name** | Название модели автомобиля (строковое) |

**Почему `mpg` — целевая переменная:** расход топлива — основной показатель экономичности автомобиля, напрямую зависящий от технических характеристик (масса, мощность, объём двигателя). Все остальные числовые признаки описывают конструкцию автомобиля и могут служить предикторами для `mpg`.

Из `describe()` видно, что средний расход — ~23.5 mpg, диапазон от 9 до 46.6. Масса варьируется от 1613 до 5140 фунтов, мощность — от 46 до 230 л.с.

In [ ]:
raw_data.describe(include='all')

## 2. Предобработка данных

Основные проблемы качества данных в этом наборе:

1. **Пропуски в `horsepower`** (6 из 398 строк). Символ `?` уже обработан при загрузке (`na_values='?'`), значения стали `NaN`. Стратегия заполнения — **медиана**: она устойчива к выбросам, в отличие от среднего, и не смещает распределение. Удаление строк тоже допустимо (потеря <2%), но заполнение медианой сохраняет больше данных.

2. **Преобразование типов.** На всякий случай приводим все числовые столбцы через `pd.to_numeric(..., errors='coerce')`, чтобы перехватить возможные скрытые строковые значения вроде `-` или `n/a`.

3. **Дубликаты.** Проверяем по числовым столбцам (разные автомобили могут иметь одинаковое название, но одинаковые технические характеристики — это дубликат).


In [ ]:
data_clean = raw_data.copy()

for col in ['mpg', 'cylinders', 'displacement', 'horsepower',
            'weight', 'acceleration', 'model_year', 'origin']:
    data_clean[col] = pd.to_numeric(data_clean[col], errors='coerce')

print(f"Пропуски до обработки:\n{data_clean.isnull().sum()}\n")

# horsepower: 6 пропусков — заменяем медианой (устойчива к выбросам)
hp_median = data_clean['horsepower'].median()
data_clean['horsepower'] = data_clean['horsepower'].fillna(hp_median)
print(f"Медиана horsepower для заполнения: {hp_median}")

# Удаление дубликатов (по всем столбцам кроме car_name)
before = len(data_clean)
data_clean = data_clean.drop_duplicates(subset=data_clean.columns.difference(['car_name']))
print(f"Удалено дубликатов: {before - len(data_clean)}")

print(f"\nПропуски после обработки:\n{data_clean.isnull().sum()}")
data_clean


**Удаление `car_name`:** столбец содержит уникальные строковые названия моделей (почти 300 уникальных значений). Использовать его в линейной регрессии напрямую невозможно, а one-hot encoding создал бы слишком много разреженных признаков при малом числе наблюдений (398). Информация о марке частично кодируется через `origin`.

**`origin` остаётся как числовой признак:** значения 1, 2, 3 имеют осмысленный порядок (США → Европа → Азия) и коррелируют с размерами/мощностью автомобилей. Для линейной регрессии такое кодирование приемлемо.

In [ ]:
clean_numeric = data_clean.drop(columns=['car_name'])
print(f"Размер: {clean_numeric.shape}")
print(f"Столбцы: {list(clean_numeric.columns)}")
clean_numeric.head()


## 3. Исследование и выбор признаков

Для оценки зависимостей строим три типа визуализаций:

- **Корреляционная heatmap** — показывает попарные линейные связи между всеми признаками. Позволяет одним взглядом увидеть, какие признаки сильно связаны с `mpg` (потенциальные предикторы) и между собой (мультиколлинеарность).
- **Диаграммы рассеяния** — для 4 наиболее коррелирующих с `mpg` признаков (`weight`, `horsepower`, `displacement`, `model_year`). Показывают форму зависимости (линейная, нелинейная) и наличие выбросов.
- **Гистограмма `mpg`** — позволяет оценить распределение целевой переменной (нормальность, скошенность, наличие выбросов).

После визуализации выбираем признаки для модели. Ключевое наблюдение: `weight`, `displacement`, `horsepower` и `cylinders` сильно коррелируют между собой (r > 0.85), что создаёт мультиколлинеарность. Можно удалить часть из них, но мы решаем оставить все 7 числовых признаков и доверить борьбу с мультиколлинеарностью Ridge-регуляризации — это позволяет не терять информацию.


In [ ]:
# --- Корреляционная матрица ---
fig, ax = plt.subplots(figsize=(10, 8))
corr = clean_numeric.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Корреляционная матрица')
plt.tight_layout()
plt.show()

print("\nКорреляция признаков с mpg:")
print(corr['mpg'].drop('mpg').sort_values())

# --- Диаграммы рассеяния ключевых признаков ---
key_features = ['weight', 'horsepower', 'displacement', 'model_year']
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, feat in zip(axes.ravel(), key_features):
    ax.scatter(clean_numeric[feat], clean_numeric['mpg'], alpha=0.5, edgecolors='k', linewidth=0.3)
    ax.set_xlabel(feat)
    ax.set_ylabel('mpg')
    ax.set_title(f'mpg vs {feat}')
plt.tight_layout()
plt.show()

# --- Распределение целевой переменной ---
fig, ax = plt.subplots(figsize=(8, 4))
clean_numeric['mpg'].hist(bins=25, ax=ax, edgecolor='black')
ax.set_xlabel('mpg')
ax.set_ylabel('Частота')
ax.set_title('Распределение mpg')
plt.tight_layout()
plt.show()


In [ ]:
# Все числовые признаки (кроме целевой) — используем полный набор,
# т.к. Ridge-регуляризация справится с мультиколлинеарностью
selected_features = ['cylinders', 'displacement', 'horsepower',
                     'weight', 'acceleration', 'model_year', 'origin']
print(f"Выбрано {len(selected_features)} признаков: {selected_features}")
selected_features


## 4. Обучающая и тестовая выборки

Разделяем данные в пропорции **80/20** (318 / 80 записей). Это стандартная пропорция: обучающая выборка достаточно велика для устойчивого обучения, а тестовая (~80 записей) — для надёжной оценки метрик.

**`random_state=42`** фиксирует случайную перестановку, обеспечивая воспроизводимость: при повторном запуске ноутбука разбиение останется тем же, а метрики — сравнимыми. Значение 42 — общепринятое по конвенции и не влияет на качество разбиения.


In [ ]:
X = clean_numeric[selected_features]
y = clean_numeric['mpg']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Обучающая выборка: {X_train.shape[0]} записей")
print(f"Тестовая выборка:  {X_test.shape[0]} записей")


## 5. Масштабирование данных

Применяем масштабирование к **признакам** и к **целевой переменной**:

**Признаки — StandardScaler** (z-нормализация: среднее → 0, стд. отклонение → 1):
- **Выравнивание шкал.** Признаки находятся на разных шкалах: `weight` ~ 1600–5100, `cylinders` ~ 3–8, `model_year` ~ 70–82. Без масштабирования Ridge-регрессия штрафовала бы коэффициенты неравномерно.
- **Интерпретируемость коэффициентов.** После StandardScaler каждый коэффициент показывает, на сколько mpg изменится при изменении признака на 1 стандартное отклонение.

**Целевая переменная — MinMaxScaler** (приведение к диапазону [0, 1]):
- Масштабирование `mpg` нормализует RMSE: вместо абсолютных значений (3–4 mpg) ошибка выражается в долях от диапазона целевой переменной, что делает метрику сравнимой между разными задачами.
- Модели обучаются предсказывать нормализованные значения, что улучшает числовую стабильность.

**Важно:** `fit_transform` применяется только к обучающей выборке, а `transform` — к тестовой. Это предотвращает утечку информации (data leakage).

In [ ]:
feature_scaler = StandardScaler()
target_scaler = MinMaxScaler()

X_train = pd.DataFrame(
    feature_scaler.fit_transform(X_train),
    columns=selected_features,
    index=X_train.index
)
X_test = pd.DataFrame(
    feature_scaler.transform(X_test),
    columns=selected_features,
    index=X_test.index
)

y_train = pd.Series(
    target_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel(),
    index=y_train.index
)
y_test = pd.Series(
    target_scaler.transform(y_test.values.reshape(-1, 1)).ravel(),
    index=y_test.index
)

print("Признаки — средние после масштабирования (train, ≈0):")
print(X_train.mean().round(6))
print(f"\ny_train после MinMaxScaler: min={y_train.min():.3f}, max={y_train.max():.3f}")


## 6. Обучение моделей

Обучаем две модели:

1. **LinearRegression (OLS)** — классический метод наименьших квадратов, без регуляризации. Служит базовой моделью (baseline) для сравнения. Прост в интерпретации: каждый коэффициент показывает вклад признака в прогноз.

2. **Ridge (alpha=1.0)** — линейная регрессия с L2-регуляризацией. Добавляет штраф $\alpha \cdot \sum \beta_i^2$ к функции потерь, что «сжимает» коэффициенты к нулю. Это особенно полезно при мультиколлинеарности: вместо того чтобы одному из коррелирующих признаков (например, `weight`) получить большой положительный коэффициент, а другому (`displacement`) — большой отрицательный, Ridge распределяет вес более равномерно и стабильно.

`alpha=1.0` — умеренная степень регуляризации (значение по умолчанию). Для более тщательного подбора можно использовать кросс-валидацию (`RidgeCV`).

После обучения выводим коэффициенты обеих моделей — это позволяет увидеть, как регуляризация перераспределяет вес между коррелирующими признаками.


In [ ]:
# Базовая линейная регрессия — отправная точка для сравнения
model_1 = LinearRegression()
model_1.fit(X_train, y_train)

# Ridge-регрессия — L2-регуляризация для борьбы с мультиколлинеарностью
# (displacement, weight, horsepower, cylinders сильно коррелируют между собой)
model_2 = Ridge(alpha=1.0)
model_2.fit(X_train, y_train)

print("Коэффициенты LinearRegression:")
for feat, coef in zip(selected_features, model_1.coef_):
    print(f"  {feat:15s} {coef:+.3f}")

print(f"\nКоэффициенты Ridge (alpha=1.0):")
for feat, coef in zip(selected_features, model_2.coef_):
    print(f"  {feat:15s} {coef:+.3f}")


## 7. Оценка моделей

Для оценки используем две метрики:

- **RMSE (Root Mean Squared Error)** — корень из среднего квадрата ошибок. Поскольку целевая переменная масштабирована MinMaxScaler в диапазон [0, 1], RMSE выражается в долях от этого диапазона. RMSE ≈ 0.08 означает, что средняя ошибка модели составляет ~8% от диапазона mpg.

- **R² (коэффициент детерминации)** — доля дисперсии целевой переменной, объяснённая моделью. R² = 0.85 означает, что модель объясняет 85% вариации в mpg, а 15% — это необъяснённый шум или влияние неучтённых факторов. Масштабирование не влияет на R².

Обе метрики вычисляются на **тестовой** выборке (80 записей), которая не участвовала в обучении. Список `models` сортируется по RMSE по возрастанию, чтобы лучшая модель была первой.


In [ ]:
def rmse(y_true, y_pred):
    return sqrt(mean_squared_error(y_true, y_pred))

y_pred_1 = model_1.predict(X_test)
y_pred_2 = model_2.predict(X_test)

rmse_1 = rmse(y_test, y_pred_1)
rmse_2 = rmse(y_test, y_pred_2)

r2_1 = r2_score(y_test, y_pred_1)
r2_2 = r2_score(y_test, y_pred_2)

print(f"{'Модель':<25s} {'RMSE':>8s} {'R²':>8s}")
print("-" * 43)
print(f"{'LinearRegression':<25s} {rmse_1:8.3f} {r2_1:8.3f}")
print(f"{'Ridge (alpha=1.0)':<25s} {rmse_2:8.3f} {r2_2:8.3f}")

models = [(model_1, rmse_1), (model_2, rmse_2)]
models.sort(key=lambda x: x[1])

print(f"\nЛучшая модель по RMSE: {type(models[0][0]).__name__} (RMSE={models[0][1]:.3f})")
models


## 8. Заключение

Ниже приведён обзор проделанной работы: основные наблюдения о наборе данных, наиболее значимые признаки и сравнение моделей.


Набор данных Auto MPG содержит 398 записей автомобилей 1970–1982 годов выпуска с 8 числовыми признаками и одной строковой переменной (название модели). Из 398 строк только 6 содержали пропущенные значения — все в столбце `horsepower`, которые были заменены медианой (93.5 л.с.), что позволило сохранить размер выборки без искажения распределения. Дубликатов в наборе данных не обнаружено.

Корреляционный анализ показал, что наибольшее влияние на расход топлива (mpg) оказывают масса автомобиля (`weight`, корреляция −0.83), рабочий объём двигателя (`displacement`, −0.80), мощность (`horsepower`, −0.77) и количество цилиндров (`cylinders`, −0.78). Эти четыре признака сильно коррелируют и между собой, что указывает на мультиколлинеарность. Год выпуска (`model_year`) положительно коррелирует с mpg (+0.58), отражая тренд на более экономичные автомобили. Признак `origin` также показывает положительную связь (+0.56) — азиатские и европейские автомобили в среднем экономичнее американских.

Для моделирования были выбраны все 7 числовых признаков. Признаки масштабированы `StandardScaler` (z-нормализация), а целевая переменная `mpg` — `MinMaxScaler` (приведение к [0, 1]). Масштабирование целевой переменной нормализует RMSE, выражая ошибку в долях от диапазона mpg, что делает метрику удобной для сравнения.

Были обучены две модели: обычная линейная регрессия (LinearRegression) и Ridge-регрессия с параметром alpha=1.0. Обе модели показали близкие результаты с R² ≈ 0.85 и RMSE ≈ 0.08 (на масштабированных данных). Наибольший вклад вносят `weight` и `model_year`, что согласуется с корреляционным анализом. Ridge-регрессия несколько стабилизирует коэффициенты при наличии мультиколлинеарности, перераспределяя вес между коррелирующими признаками, но разница в итоговых метриках минимальна — при 7 признаках и 398 наблюдениях переобучение не критично.

Итоговый R² ≈ 0.85 означает, что модели объясняют около 85 % дисперсии в расходе топлива. Для повышения точности можно попробовать нелинейные модели (Random Forest, Gradient Boosting) или добавить производные признаки (например, отношение мощности к массе).